<a href="https://colab.research.google.com/github/somendrew/LangGraph_tutorial/blob/main/6_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. What is it?

A cycle (loop) means the graph can go back to a previous node and repeat execution until some condition is satisfied.


---


### 2. Why does it matter?

You use loops when:

You need retry logic</br>
You want to improve output step-by-step</br>
You need iteration until success</br>

Without loops → your agent is one-pass only</br>
With loops → your agent becomes intelligent & self-correcting


---


### 3. Real-world analogy

Think of cooking:

Taste food 🍲</br>
Too salty? → fix it</br>
Taste again</br>
Repeat until perfect</br>

That “taste → fix → repeat” = loop


---


### 4. Visual Diagram (Core Idea)

        ┌──────────────┐
        │   generate   │
        └──────┬───────┘
               │
               ▼
        ┌──────────────┐
        │   validate   │
        └──────┬───────┘
         ┌─────┴─────┐
     good ✅      bad ❌
         │             │
         ▼             │
       END             │
                       │
                       ▼
                   generate

In [3]:
!pip install -q langgraph Typing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# ----------------------------
# 1. State
# ----------------------------
class State(TypedDict):
    user_query: str
    answer: str
    feedback: str
    attempts: int

# ----------------------------
# 2. Generate initial answer
# ----------------------------
def generate(state: State) -> State:
    attempts = state["attempts"] + 1

    # Simulate weak first answer
    if attempts == 1:
        answer = "Contact support."
    else:
        answer = state["answer"]  # keep improved version

    return {**state, "answer": answer, "attempts": attempts}

# ----------------------------
# 3. Critique answer
# ----------------------------
def critique(state: State) -> State: # Changed return type to State
    answer = state["answer"]
    critique_result = "bad" if len(answer) < 30 else "good" # Store result
    return {**state, "feedback": critique_result} # Return updated state

# ----------------------------
# 4. Improve answer
# ----------------------------
def improve(state: State) -> State:
    improved = state["answer"] + " Please provide your order ID so we can assist you faster."

    return {
        **state,
        "answer": improved,
        "feedback": "Improved answer"
    }

# ----------------------------
# 5. Build graph
# ----------------------------
builder = StateGraph(State)

builder.add_node("generate", generate)
builder.add_node("critique", critique)
builder.add_node("improve", improve)

builder.set_entry_point("generate")

# Flow
builder.add_edge("generate", "critique")
builder.add_edge("improve", "critique")  # NEW: loop path continues

# ----------------------------
# 6. Conditional loop
# ----------------------------
builder.add_conditional_edges(
    "critique",
    lambda state: state["feedback"], # Use lambda to read feedback from state
    {
        "good": END,
        "bad": "improve"  # LOOP here
    }
)

# ----------------------------
# 7. Compile & run
# ----------------------------
graph = builder.compile()

result = graph.invoke({
    "user_query": "Where is my order?",
    "answer": "",
    "feedback": "",
    "attempts": 0
})

print(result["answer"])

Contact support. Please provide your order ID so we can assist you faster.
